# 01 — Reuse Albert WC Response Assets

Parse `smeft_contents.txt`, discover usable coffea files/cards, and build a manifest used by fitting notebooks.

In [ ]:
from pathlib import Path
import json, sys, os, math
import numpy as np
import pandas as pd

def find_repo_root(start=None, marker='smeft_contents.txt', max_up=8):
    p = Path(start or Path.cwd()).resolve()
    for _ in range(max_up+1):
        if (p / marker).exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise FileNotFoundError(f'Could not find repo root containing {marker} from {Path.cwd()}')

REPO = find_repo_root()
NOTEBOOK_DIR = REPO / 'notebooks_hww_fit'
SMEFT_LISTING = REPO / 'smeft_contents.txt'
LOCAL_DICT = NOTEBOOK_DIR / 'dictionary.json'
print('repo:', REPO)
paths=[l.strip() for l in SMEFT_LISTING.read_text().splitlines() if l.strip()]

In [ ]:
analysis_paths=[p for p in paths if '/uscms_data/d3/azhou/smeft/analysis/' in p]
coffea_files=[p for p in paths if p.endswith('.coffea')]
reweight_cards=[p for p in paths if p.endswith('_reweight_card.dat')]
dict_candidates=[p for p in paths if p.endswith('dictionary.json')]
print('analysis entries',len(analysis_paths))
print('coffea files',len(coffea_files))
print('reweight cards',len(reweight_cards))
print('dictionary candidates',len(dict_candidates))

In [ ]:
# Heuristic selection for defaults; edit as needed
def pick_first(patterns, pool):
    for pat in patterns:
        for p in pool:
            if pat in p:
                return p
    return pool[0] if pool else None

default_card = pick_first(['VBF_SMEFTsim_topU3l_NP1_reweight_card.dat'], reweight_cards)
default_coffea = pick_first(['start0.coffea','VBF_SMEFTsim_topU3l_NP1.coffea'], coffea_files)
manifest={
 'default_reweight_card': default_card,
 'default_coffea': default_coffea,
 'coffea_files': coffea_files[:200],
 'reweight_cards': reweight_cards[:200],
 'dictionary_path': str((NOTEBOOK_DIR/'dictionary.json').resolve())
}
(NOTEBOOK_DIR/'asset_manifest.json').write_text(json.dumps(manifest, indent=2))
print('default card:', default_card)
print('default coffea:', default_coffea)
print('wrote asset_manifest.json')